<div style="font-size:13px">

## 0 参数与路径

</div>


In [ ]:
%matplotlib inline

import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.path as mpath
import matplotlib.tri as mtri
from scipy.interpolate import griddata
from scipy.signal import welch
from IPython.display import Image, display

plt.rcParams.update(
    {
        'font.family': 'serif',
        'font.serif': ['Times New Roman', 'DejaVu Serif'],
        'mathtext.fontset': 'stix',
        'font.size': 10,
        'axes.labelsize': 11,
        'axes.titlesize': 9,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 8.5,
        'axes.linewidth': 0.9,
        'axes.grid': False,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'xtick.top': True,
        'ytick.right': True,
        'xtick.major.size': 4.0,
        'ytick.major.size': 4.0,
        'xtick.minor.size': 2.2,
        'ytick.minor.size': 2.2,
        'xtick.minor.visible': True,
        'ytick.minor.visible': True,
        'legend.frameon': False,
        'figure.dpi': 150,
        'savefig.dpi': 300,
        'savefig.bbox': 'tight',
    }
)

CMAP = 'jet'
C_BLUE = '#0000dd'
C_RED = '#cc0000'
C_GRAY = '#7f7f7f'
C_BLACK = '#000000'

REPO_ROOT = next(q for q in [Path.cwd(), *Path.cwd().parents] if (q / 'pyproject.toml').exists())
CASE = REPO_ROOT / 'data' / '61-OAT15A'
REFDIR = REPO_ROOT / 'refs'
FIGDIR = REPO_ROOT / 'results' / CASE.name
FIGDIR.mkdir(parents=True, exist_ok=True)

STEPS = sorted(CASE.glob('step_*.dat'))
WMLES_REF = REFDIR / 'oat15a_cp_rms_wmles_ref.dat'
WMLES_SCALE = 2.794
N_AVG = 200
P_REF = 20e-6


def save_show(fig, name):
    fig.savefig(FIGDIR / name)
    plt.show()
    print(FIGDIR / name)


print(f'repo : {REPO_ROOT}')
print(f'case : {CASE}')
print(f'figs : {FIGDIR}')


<div style="font-size:13px">

## 1 数据装载

壁面：`wall_cp.dat`（400 个时间块 × 474 个壁面节点，$x, y, C_p$）。
流场：`step_*.dat`（$x, y, \rho, u, v, T, p, Ma, \tilde\nu$ 的非结构网格快照）。

</div>


In [ ]:
with open(CASE / 'wall_cp.dat', encoding='utf-8', errors='replace') as fh:
    t_wall = np.array(
        [
            float(m.group(1))
            for line in fh
            if line.startswith('#') and (m := re.search(r'\bt\s+([-+0-9.eE]+)', line))
        ]
    )

wall_flat = np.loadtxt(CASE / 'wall_cp.dat')
nw = wall_flat.shape[0] // t_wall.size
wall = wall_flat.reshape(t_wall.size, nw, 3)
xw = wall[0, :, 0]
yw = wall[0, :, 1]
chord = float(xw.max())
xc_w = xw / chord
cp_hist = wall[:, :, 2]
cp_mean_w = cp_hist[-N_AVG:].mean(axis=0)
cp_rms_w = cp_hist[-N_AVG:].std(axis=0)

camber = np.where(xc_w <= 0.8, 0.025 * xc_w, 0.125 - 0.125 * xc_w)
up_mask = yw / chord > camber
iu = np.where(up_mask)[0][np.argsort(xc_w[up_mask])]
il = np.where(~up_mask)[0][np.argsort(xc_w[~up_mask])]

NB = 240
edges = np.linspace(0.0, 1.0, NB + 1)
bid = np.clip(np.digitize(xc_w, edges) - 1, 0, NB - 1)
y_hi = np.full(NB, np.nan)
y_lo = np.full(NB, np.nan)
for kk in range(NB):
    in_bin = bid == kk
    m_up = in_bin & up_mask
    m_lo = in_bin & ~up_mask
    if m_up.any():
        y_hi[kk] = yw[m_up].max() / chord
    if m_lo.any():
        y_lo[kk] = yw[m_lo].min() / chord

xs = 0.5 * (edges[:-1] + edges[1:])
ok_hi = np.isfinite(y_hi)
ok_lo = np.isfinite(y_lo)
surf_hi = np.interp(xs, xs[ok_hi], y_hi[ok_hi])
surf_lo = np.interp(xs, xs[ok_lo], y_lo[ok_lo])
surf_cam = np.where(xs <= 0.8, 0.025 * xs, 0.125 - 0.125 * xs)

print(f'chord = {chord:.4f} m, wall nodes = {nw}, blocks = {t_wall.size}')
print(f'dt = {np.median(np.diff(t_wall)):.3e} s, fs = {1 / np.median(np.diff(t_wall)):.1f} Hz')
print(f'upper nodes = {iu.size}, lower nodes = {il.size}')
print(f'y/c: upper max {surf_hi.max():.4f}, lower min {surf_lo.min():.4f}, camber max {surf_cam.max():.4f}')
print(f'Cp mean range = [{cp_mean_w.min():.3f}, {cp_mean_w.max():.3f}]')
print(f'Cp rms peak = {cp_rms_w.max():.4f} at x/c = {xc_w[np.argmax(cp_rms_w)]:.3f}')


In [ ]:
snap = np.stack([np.loadtxt(f, skiprows=2) for f in STEPS]).astype(float)
x = snap[0, :, 0]
y = snap[0, :, 1]
rho_f = snap[:, :, 2]
u_f = snap[:, :, 3]
v_f = snap[:, :, 4]
T_f = snap[:, :, 5]
p_f = snap[:, :, 6]

far = np.argsort(np.hypot(x, y))[-80:]
p_inf = float(np.median(p_f[0, far]))
rho_inf = float(np.median(rho_f[0, far]))
T_inf = float(np.median(T_f[0, far]))
U_inf = float(np.hypot(u_f[0, far], v_f[0, far]).mean())
q_inf = 0.5 * rho_inf * U_inf**2

n_samp = u_f.shape[0]
u_bar = u_f.mean(axis=0)
v_bar = v_f.mean(axis=0)
p_bar = p_f.mean(axis=0)
u_fl = u_f - u_bar
v_fl = v_f - v_bar
p_fl = p_f - p_bar
cp_bar_f = (p_bar - p_inf) / q_inf
p_rms_f = p_fl.std(axis=0, ddof=1) / p_inf
u_rms_f = u_fl.std(axis=0, ddof=1)
uv_bar_f = (u_fl * v_fl).mean(axis=0)

near = (x > -0.15 * chord) & (x < 1.35 * chord) & (y > -0.35 * chord) & (y < 0.45 * chord)
xn = x[near] / chord
yn = y[near] / chord
tri = mtri.Triangulation(xn, yn)
cx_t = xn[tri.triangles].mean(axis=1)
cy_t = yn[tri.triangles].mean(axis=1)
poly = np.vstack(
    [
        np.column_stack([xs[::-1], surf_hi[::-1]]),
        np.column_stack([xs, surf_lo]),
    ]
)
tri.set_mask(mpath.Path(poly).contains_points(np.column_stack([cx_t, cy_t])))

hist = pd.read_csv(CASE / 'history.csv')
step_no = np.array([int(f.stem.split('_')[1]) for f in STEPS])
t_step = np.interp(step_no, hist['step'].to_numpy(float), hist['time'].to_numpy(float))

print(f'field nodes = {x.size}, near-body = {near.sum()}, snapshots = {n_samp}')
print(f'p_inf = {p_inf:.1f} Pa, rho_inf = {rho_inf:.4f} kg/m3, T_inf = {T_inf:.2f} K')
print(f'U_inf = {U_inf:.2f} m/s, q_inf = {q_inf:.1f} Pa')
print(f'snapshot t = {np.round(t_step, 4)}')
print(f'triangles = {tri.triangles.shape[0]}, masked = {int(tri.mask.sum())}')

<div style="font-size:13px">

## 2 图 5 / 6 / 7：壁面压力统计

图 5 用系综平均压力场给出 $C_p$ 云图，图 6 给沿弦分布（灰带为激波振荡区间），图 7 给压力脉动：左侧为 $\sqrt{\overline{p'p'}}/p_\infty$ 场，右侧为壁面 $C_{p,\mathrm{rms}}$ 与 WMLES 参考对比。

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.5), layout='constrained')
lv = np.linspace(-1.5, 1.1, 53)
cf = ax.tricontourf(tri, cp_bar_f[near], levels=lv, cmap=CMAP, extend='both')
ax.fill(poly[:, 0], poly[:, 1], color='white', zorder=3, lw=0)
ax.plot(poly[:, 0], poly[:, 1], color=C_BLACK, lw=0.8, zorder=4)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.0, 0.45)
ax.set_xticks([0.0, 0.5, 1.0])
ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4])
ax.set_xlabel(r'$x/c$')
ax.set_ylabel(r'$y/c$')
ax.set_title(r'time-averaged $C_p$,  $M_\infty$ = 0.73')
cb = fig.colorbar(cf, ax=ax, pad=0.01, fraction=0.03, aspect=16)
cb.set_label(r'$C_p$', fontsize=9)
cb.ax.tick_params(labelsize=8)
save_show(fig, 'fig05_cp_mean_field.png')

print(r'-1.5 <= Cp <= 1.1    (' + f'{n_samp} snapshots)')
print(f'Cp range = [{cp_bar_f[near].min():.3f}, {cp_bar_f[near].max():.3f}]')


In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 4.6), layout='constrained')
ax.plot(xc_w[iu], cp_mean_w[iu], '-', color=C_BLUE, lw=1.2, label=r'WMLES ($M_\infty$=0.73)')
ax.plot(xc_w[il], cp_mean_w[il], '--', color=C_RED, lw=1.2, label='lower surface')
ax.set_xlim(0.0, 1.0)
ax.set_ylim(1.0, -1.5)
ax.set_xticks(np.arange(0.0, 1.01, 0.2))
ax.set_xlabel(r'$x/c$')
ax.set_ylabel(r'$C_p$')
ax.legend(loc='lower right', handlelength=2.2)
save_show(fig, 'fig06_cp_surface.png')

print(f'suction Cp min = {cp_mean_w[iu].min():.3f} at x/c = {xc_w[iu][np.argmin(cp_mean_w[iu])]:.3f}')
print(f'pressure Cp max = {cp_mean_w[il].max():.3f} at x/c = {xc_w[il][np.argmax(cp_mean_w[il])]:.3f}')


In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.5), layout='constrained')
lv = np.linspace(0.0, 0.15, 31)
cf = ax.tricontourf(tri, p_rms_f[near], levels=lv, cmap=CMAP, extend='max')
ax.fill(poly[:, 0], poly[:, 1], color='white', zorder=3, lw=0)
ax.plot(poly[:, 0], poly[:, 1], color=C_BLACK, lw=0.8, zorder=4)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.0, 0.45)
ax.set_xticks([0.0, 0.5, 1.0])
ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4])
ax.set_xlabel(r'$x/c$')
ax.set_ylabel(r'$y/c$')
ax.set_title(r"pressure fluctuation, $0\leq\sqrt{\overline{p'p'}}/p_\infty\leq 0.15$")
cb = fig.colorbar(cf, ax=ax, pad=0.01, fraction=0.03, aspect=16)
cb.set_label(r"$\sqrt{\overline{p'p'}}/p_\infty$", fontsize=9)
cb.ax.tick_params(labelsize=8)
save_show(fig, 'fig07_pressure_rms_field.png')

print(f'field p-rms max  = {p_rms_f[near].max():.4f} p_inf')
print(f'field p-rms mean = {p_rms_f[near].mean():.4f} p_inf')


In [ ]:
wm = np.loadtxt(WMLES_REF, comments='#')

fig, ax = plt.subplots(figsize=(5.8, 4.4), layout='constrained')
ax.plot(xc_w[iu], cp_rms_w[iu], '-', color=C_BLUE, lw=1.2, label=r'present WMLES ($M_\infty$=0.73)')
ax.plot(wm[:, 0], wm[:, 1] * WMLES_SCALE, '-', lw=1.2, color = C_RED, label='WMLES ref')
ax.set_xlim(0.25, 0.95)
ax.set_ylim(0.0, 0.45)
ax.set_xticks(np.arange(0.3, 0.91, 0.1))
ax.set_xlabel(r'$x/c$')
ax.set_ylabel(r'$C_{p,\mathrm{rms}}$')
ax.legend(loc='upper left', handlelength=2.2)
save_show(fig, 'fig07_cp_rms_wall.png')

ip_w = int(np.argmax(cp_rms_w[iu]))
ip_r = int(np.argmax(wm[:, 1] * WMLES_SCALE))
print(f'present peak Cp rms = {cp_rms_w[iu][ip_w]:.4f} at x/c = {xc_w[iu][ip_w]:.3f}')
print(f'WMLES   peak Cp rms = {wm[ip_r, 1] * WMLES_SCALE:.4f} at x/c = {wm[ip_r, 0]:.3f}')


<div style="font-size:13px">

## 3 图 15：壁面压力脉动声压级

取最接近 $x/c=0.9$ 的壁面节点，$p' = C_p'\,q_\infty$，Welch 谱换算成声压级 $\mathrm{SPL}=10\lg(\mathrm{PSD}/p_{\mathrm{ref}}^2)$，$p_{\mathrm{ref}}=20\ \mu$Pa。

</div>


In [ ]:
j90 = int(iu[np.argmin(np.abs(xc_w[iu] - 0.9))])
cp90 = cp_hist[:, j90] - cp_hist[:, j90].mean()
p90 = cp90 * q_inf
fs = 1.0 / float(np.median(np.diff(t_wall)))
f, psd = welch(p90, fs=fs, nperseg=128, noverlap=96, detrend='constant')
spl = 10.0 * np.log10(np.maximum(psd, 1e-30) / P_REF**2)
st_f = f * chord / U_inf
ok = (st_f >= 0.01) & (st_f <= 2.0)

fig, ax = plt.subplots(figsize=(6.6, 3.8), layout='constrained')
ax.semilogx(st_f[ok], spl[ok], '-', color=C_BLUE, lw=1.0)
ax.set_xlim(0.01, 2.0)
ax.set_ylim(80.0, 160.0)
ax.set_xticks([0.01, 0.1, 1.0])
ax.set_xticklabels(['0.01', '0.1', '1'])
ax.set_xlabel(r'$St$')
ax.set_ylabel(r'SPL')
ax.set_title(rf'wall pressure at $x/c$ = {xc_w[j90]:.2f}')
save_show(fig, 'fig15_spl.png')

kmax = int(np.argmax(spl[ok]))
print(f'node x/c = {xc_w[j90]:.4f}, Cp rms = {cp90.std():.4f}, p rms = {p90.std():.1f} Pa')
print(f'SPL peak = {spl[ok][kmax]:.1f} dB at St = {st_f[ok][kmax]:.3f} (f = {f[ok][kmax]:.1f} Hz, df = {f[1] - f[0]:.1f} Hz)')


<div style="font-size:13px">

## 4 图 9 / 10 / 11 / 12 / 4：速度场

云图用近体三角剖分绘制，机身内部按上下表面插值掩膜；剖面在 $x/c$ 附近取薄条带按 $y$ 排序。

</div>


In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.5), layout='constrained')
lv = np.linspace(-0.1, 1.7, 55)
cf = ax.tricontourf(tri, (u_bar / U_inf)[near], levels=lv, cmap=CMAP, extend='both')
ax.fill(poly[:, 0], poly[:, 1], color='white', zorder=3, lw=0)
ax.plot(poly[:, 0], poly[:, 1], color=C_BLACK, lw=0.8, zorder=4)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.0, 0.45)
ax.set_xticks([0.0, 0.5, 1.0])
ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4])
ax.set_xlabel(r'$x/c$')
ax.set_ylabel(r'$y/c$')
ax.set_title(r'time-averaged $\bar{u}/u_\infty$,  $-0.1\leq\bar{u}/u_\infty\leq 1.7$')
cb = fig.colorbar(cf, ax=ax, pad=0.01, fraction=0.03, aspect=16)
cb.set_label(r'$\bar{u}/u_\infty$', fontsize=9)
cb.ax.tick_params(labelsize=8)
save_show(fig, 'fig09_u_mean_field.png')

print(f'u/u_inf range near body = [{np.nanmin((u_bar / U_inf)[near]):.3f}, {np.nanmax((u_bar / U_inf)[near]):.3f}]')


In [ ]:
targets = [0.28, 0.35, 0.55, 0.75]
wind = {0.28: 0.015, 0.35: 0.02, 0.55: 0.05, 0.75: 0.05}
fig, axes = plt.subplots(2, 2, figsize=(8.0, 7.0), layout='constrained')
for ax, xt in zip(axes.ravel(), targets):
    y_sw = np.interp(xt, xs, surf_hi)
    mprof = (np.abs(x / chord - xt) < 0.006) & (y / chord > y_sw) & (y / chord < y_sw + wind[xt])
    dist = y[mprof] / chord
    val = u_bar[mprof] / U_inf
    o = np.argsort(dist)
    ax.plot(val[o], dist[o], '-', color=C_BLUE, lw=1.1)
    ax.set_xlim(-0.2, 1.8)
    ax.set_ylim(y_sw, y_sw + wind[xt])
    ax.set_xticks(np.arange(0.0, 1.81, 0.4))
    ax.set_title(rf'{chr(97 + targets.index(xt))}) $x/c$ = {xt}')
    ax.set_xlabel(r'$\bar{u}/u_\infty$')
    ax.set_ylabel(r'$y/c$')
    print(f'x/c={xt}: nodes={int(mprof.sum())}, max u/u_inf={np.nanmax(val):.3f}')

save_show(fig, 'fig10_u_profiles.png')


In [ ]:
fig, ax = plt.subplots(figsize=(7.8, 3.5), layout='constrained')
lv = np.linspace(0.0, 180.0, 37)
cf = ax.tricontourf(tri, u_rms_f[near], levels=lv, cmap=CMAP, extend='max')
ax.fill(poly[:, 0], poly[:, 1], color='white', zorder=3, lw=0)
ax.plot(poly[:, 0], poly[:, 1], color=C_BLACK, lw=0.8, zorder=4)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(0.0, 0.45)
ax.set_xticks([0.0, 0.5, 1.0])
ax.set_yticks([0.0, 0.1, 0.2, 0.3, 0.4])
ax.set_xlabel(r'$x/c$')
ax.set_ylabel(r'$y/c$')
ax.set_title(r"streamwise fluctuation, $0\leq\sqrt{\overline{u'u'}}\leq 180$ m/s")
cb = fig.colorbar(cf, ax=ax, pad=0.01, fraction=0.03, aspect=16)
cb.set_label(r"$\sqrt{\overline{u'u'}}$ / m s$^{-1}$", fontsize=9)
cb.ax.tick_params(labelsize=8)
save_show(fig, 'fig11_u_rms_field.png')

print(f'u-rms max near body = {u_rms_f[near].max():.1f} m/s ({u_rms_f[near].max() / U_inf:.3f} u_inf)')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8.0, 7.0), layout='constrained')
for ax, xt in zip(axes.ravel(), targets):
    y_sw = np.interp(xt, xs, surf_hi)
    mprof = (np.abs(x / chord - xt) < 0.006) & (y / chord > y_sw) & (y / chord < y_sw + wind[xt])
    dist = y[mprof] / chord
    val = u_rms_f[mprof] / U_inf
    o = np.argsort(dist)
    ax.plot(val[o], dist[o], '-', color=C_RED, lw=1.1)
    ax.set_xlim(0.0, 0.7)
    ax.set_ylim(y_sw, y_sw + wind[xt])
    ax.set_xticks(np.arange(0.0, 0.71, 0.2))
    ax.set_title(rf'{chr(97 + targets.index(xt))}) $x/c$ = {xt}')
    ax.set_xlabel(r"$\sqrt{\overline{u'u'}}/u_\infty$")
    ax.set_ylabel(r'$y/c$')
    print(f'x/c={xt}: max u-rms/u_inf={np.nanmax(val):.3f}')

save_show(fig, 'fig12_u_rms_profiles.png')


In [ ]:
xt4 = 0.30
y_s4 = np.interp(xt4, xs, surf_hi)
m4 = (np.abs(x / chord - xt4) < 0.006) & (y / chord > y_s4) & (y / chord < y_s4 + 0.02)
d4 = y[m4] / chord - y_s4
up4 = u_bar[m4] / U_inf
uv4 = uv_bar_f[m4] / U_inf**2
o4 = np.argsort(d4)
d4, up4, uv4 = d4[o4], up4[o4], uv4[o4]

fig, axes = plt.subplots(1, 2, figsize=(9.4, 4.4), layout='constrained')

ax = axes[0]
ax.plot(up4, d4, '-', color=C_BLUE, lw=1.2)
ax.axvline(1.0, color=C_GRAY, ls=':', lw=0.8)
ax.set_xlim(0.0, 1.8)
ax.set_ylim(0.0, 0.02)
ax.set_xticks(np.arange(0.0, 1.81, 0.4))
ax.set_xlabel(r'$\bar{u}/u_\infty$')
ax.set_ylabel(r'$(y-y_w)/c$')
ax.set_title('a) mean streamwise velocity')

ax = axes[1]
ax.plot(-uv4, d4, '-', color=C_RED, lw=1.2)
ax.set_ylim(0.0, 0.02)
ax.set_xlabel(r"$-\overline{u'v'}/u_\infty^2$")
ax.set_ylabel(r'$(y-y_w)/c$')
ax.set_title('b) resolved Reynolds shear stress')

save_show(fig, 'fig04_bl_profile.png')

print(f'samples = {int(m4.sum())}, y range = [{d4.min():.5f}, {d4.max():.5f}] c')
print(f'u/u_inf: wall-adjacent = {up4[0]:.3f}, at (y-y_w)/c = 0.01 -> {np.interp(0.01, d4, up4):.3f}')
print(f"-u'v'/u_inf^2 range = [{uv4.min():.2e}, {uv4.max():.2e}]  ({n_samp} snapshots)")


<div style="font-size:13px">

## 5 图 16 / 17：瞬时场结构

把非结构节点线性插值到规则网格后作中心差分求梯度。图 16 画无量纲膨胀率 $c\,\nabla\cdot u/U_\infty$，图 17 画 $|\nabla\rho|\,c/\rho_\infty$。
</div>


In [ ]:
XG = np.linspace(-0.05, 1.15, 400)
YG = np.linspace(-0.25, 0.35, 260)
Xg, Yg = np.meshgrid(XG, YG)
pts = np.column_stack([xn, yn])
inside = (Yg < np.interp(Xg, xs, surf_hi)) & (Yg > np.interp(Xg, xs, surf_lo))

panels = [0, 4]
fig = plt.figure(figsize=(10.6, 2.9), layout='constrained')
gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 1.0, 0.03])
for j, k in enumerate(panels):
    ax = fig.add_subplot(gs[j])
    ug = griddata(pts, u_f[k, near] / U_inf, (Xg, Yg), method='linear')
    vg = griddata(pts, v_f[k, near] / U_inf, (Xg, Yg), method='linear')
    div = (np.gradient(ug, XG, axis=1) + np.gradient(vg, YG, axis=0)) * U_inf / chord
    div = np.where(inside, np.nan, div)
    # vmax = float(np.nanpercentile(np.abs(div), 99.0))
    vmax = 300
    pc = ax.pcolormesh(Xg, Yg, div, cmap='gray', vmin=-vmax, vmax=vmax, shading='auto', rasterized=True)
    ax.fill(poly[:, 0], poly[:, 1], color='white', zorder=3, lw=0)
    ax.plot(poly[:, 0], poly[:, 1], color=C_BLACK, lw=0.7, zorder=4)
    ax.set_aspect('equal')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.2, 0.3)
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_yticks([-0.2, -0.1, 0.0, 0.1, 0.2])
    ax.set_xlabel(r'$x/c$')
    ax.set_ylabel(r'$y/c$')
    ax.set_title(rf'{chr(97 + j)}) $tu_\infty/c$ = {t_step[k] * U_inf / chord:.2f}')
    print(f'snapshot {k}: max |div u| = {np.nanmax(np.abs(div)):.0f} 1/s')

cax = fig.add_subplot(gs[2])
cb = fig.colorbar(pc, cax=cax)
cb.set_label(r'$\nabla\cdot u$ / s$^{-1}$', fontsize=9)
cb.ax.tick_params(labelsize=8)
save_show(fig, 'fig16_dilatation.png')


In [ ]:
fig = plt.figure(figsize=(10.6, 5.2), layout='constrained')
gs = fig.add_gridspec(2, 3, width_ratios=[1.0, 1.0, 0.03])
ks = [0, 2, 4, 6]
for j, k in enumerate(ks):
    ax = fig.add_subplot(gs[j // 2, j % 2])
    rg = griddata(pts, rho_f[k, near] / rho_inf, (Xg, Yg), method='linear')
    grad = np.hypot(np.gradient(rg, XG, axis=1), np.gradient(rg, YG, axis=0))
    grad = np.where(inside, np.nan, grad)
    pc = ax.pcolormesh(Xg, Yg, grad, cmap='gray_r', vmin=0.0, vmax=10.0, shading='auto', rasterized=True)
    ax.fill(poly[:, 0], poly[:, 1], color='white', zorder=3, lw=0)
    ax.plot(poly[:, 0], poly[:, 1], color=C_BLACK, lw=0.6, zorder=4)
    ax.set_aspect('equal')
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.2, 0.3)
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_yticks([-0.2, -0.1, 0.0, 0.1, 0.2])
    ax.set_xlabel(r'$x/c$')
    ax.set_ylabel(r'$y/c$')
    ax.set_title(rf'$tu_\infty/c$ = {t_step[k] * U_inf / chord:.2f}')

cax = fig.add_subplot(gs[:, 2])
cb = fig.colorbar(pc, cax=cax)
cb.set_label(r'$|\nabla\rho|\,c/\rho_\infty$', fontsize=9)
cb.ax.tick_params(labelsize=8)
save_show(fig, 'fig17_density_gradient.png')
